# Lab03 — Train đầy đủ theo epoch, cấu hình regularization mới

Mỗi cấu hình chạy **đủ 10 epoch**, train toàn bộ 8.530 mẫu và validation đủ 1.066 mẫu mỗi epoch. Console có đúng một dòng tổng kết `Epoch 01/10 ... Epoch 10/10`. Biểu đồ chính có đúng **10 điểm** ở trục Epoch 1–10. Test chỉ được đánh giá sau khi chốt model và threshold bằng validation.

Đây là **thí nghiệm mới chưa có kết quả GPU**, nhằm giảm overfitting đã quan sát ở study trước. Hai candidate dùng LR `5e-6` và `1e-5`, weight decay `0.1`, encoder/attention dropout `0.2`, classifier dropout `0.3`. Vẫn dùng cross-entropy gốc, không label smoothing, nên không đánh đồng loss khác công thức. Không hứa validation loss sẽ giảm liên tục hoặc accuracy chắc chắn tăng.

| Phase | Công việc | Module tham chiếu | Output và visualize |
|---|---|---|---|
| P0 | Clone, venv Python 3.12, GPU T4 + Internet | requirements.txt, config/regularized.json | môi trường và GPU |
| P1 | Pytest + sanity overfit mô hình nhỏ | tests/test_epoch_training.py | XML/log preflight dưới outputs/ |
| P2 | Tokenization, pipeline/manual sentiment | src/models/classifier.py, src/analysis.py | token heatmap, probabilities |
| P3 | Full official splits và overlap audit | src/datasets/loader.py, src/study.py | class counts, token histogram |
| P4 | Baseline fit train, chọn C bằng validation | src/study.py | baseline candidates |
| P5 | Hai LR, mỗi LR đủ 10 epoch | src/training/trainer.py | epoch_history.json, 10 dòng log, curves 10 điểm |
| P6 | Chọn best epoch và threshold bằng validation | src/training/thresholds.py, config/regularized_study.json | frozen selection và threshold curves |
| P7 | Test sau khi khóa lựa chọn | src/study.py, src/study_evidence.py | confusion matrix, metrics, ROC/PR, calibration |
| P8 | Hiển thị trực tiếp và lưu artifact | scripts/display_study.py | ảnh inline chính là file PNG đã lưu |
| P9 | Audit và đóng gói | scripts/audit_study.py, scripts/package_study.py | review ZIP và checkpoint ZIP riêng |

Bật GPU + Internet trước khi Run All. Chỉ dùng GPU đầu tiên dù chọn T4 x2. Hai candidate × 10 epoch = tổng 20 epoch, mỗi candidate khởi tạo lại từ cùng pretrained backbone/seed. Xem [walkthrough](../walkthrough_lab_03.md), [description](../description_lab_03.md) và [kết quả study cũ](05_study_results_and_discussion.ipynb).

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink

REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
BRANCH = "lab03"
REPO = Path("/kaggle/working/Lab-DL")
ENV = Path("/kaggle/working/lab03-py312")
PYTHON = str(ENV / "bin/python")

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Hãy bật Settings > Accelerator > GPU và chạy lại từ cell đầu.")
subprocess.run(["nvidia-smi"], check=True)

if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    print("Repo đã tồn tại; giữ nguyên file và kết quả hiện có.")
LAB = REPO / "labs/lab03"
assert (LAB / "config/regularized.json").is_file(), "Repo chưa có cấu hình epoch mới; cập nhật branch lab03 trước khi chạy."
print("Code revision:")
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.5"], check=True)
def uv(*args):
    subprocess.run([sys.executable, "-m", "uv", *map(str, args)], check=True)
uv("python", "install", "3.12")
if not Path(PYTHON).is_file():
    uv("venv", "--python", "3.12", ENV)
uv("pip", "install", "--python", PYTHON, "--link-mode", "copy", "-r", LAB / "requirements.txt")
uv("pip", "check", "--python", PYTHON)

RUN_ENV = os.environ.copy()
RUN_ENV.update(CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
               CUBLAS_WORKSPACE_CONFIG=":4096:8", MPLCONFIGDIR="/kaggle/working/.mpl-lab03")
def run_python(*args, env=None):
    return subprocess.run([PYTHON, "-u", *map(str, args)], cwd=REPO,
                          env=RUN_ENV if env is None else env, check=True)
run_python("-c", "import sys, torch, transformers; print(sys.version); print('torch', torch.__version__); print('transformers', transformers.__version__); assert torch.cuda.is_available(), 'GPU chưa khả dụng'; print(torch.cuda.get_device_name(0))")
free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Dung lượng trống: {free_gib:.1f} GiB")
if free_gib < 6:
    raise RuntimeError("Cần ít nhất 6 GiB trống cho candidate checkpoints và ZIP. Hãy dùng session sạch.")

## P1 — Kiểm thử trước khi dùng GPU

Các tests dùng mô hình DistilBERT cực nhỏ và dữ liệu tổng hợp; đây là kiểm thử pipeline, không phải benchmark. Cell này ép CPU cho tests, có bài kiểm tra đủ 10 epoch trên fixture nhỏ và sanity overfit. Mọi test phải pass trước khi chạy study. Pytest XML và log được lưu vào `outputs/`.

In [ ]:
from datetime import datetime, timezone
validation_dir = LAB / "outputs/preflight" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_dir.mkdir(parents=True, exist_ok=False)
TEST_ENV = RUN_ENV.copy()
TEST_ENV["CUDA_VISIBLE_DEVICES"] = ""
with (validation_dir / "pytest.log").open("w", encoding="utf-8") as log:
    process = subprocess.Popen([PYTHON, "-u", "-m", "pytest", "labs/lab03/tests", "-q",
                                f"--junitxml={validation_dir / 'pytest.xml'}"],
                               cwd=REPO, env=TEST_ENV, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end=""); log.write(line)
    if process.wait() != 0:
        raise RuntimeError("Preflight tests failed. Chưa chạy training; xem pytest.log.")
print("Preflight passed:", validation_dir)

## P2 — Exercise 1: pretrained sentiment analysis

Tokenize một câu, đối chiếu pipeline với manual forward/softmax, rồi hiển thị ảnh từ chính artifacts đã lưu. Không dùng mô hình SST-2 này làm backbone cho Exercise 2: Exercise 2 khởi tạo classification head mới từ DistilBERT base.

In [ ]:
run_python("labs/lab03/scripts/run_exercise1.py")
run_python("-c", "from labs.lab03.src.analysis import ensure_exercise1_analysis; from labs.lab03.src.utils.runs import ROOT; import json; s=json.loads((ROOT/'config/workflow.json').read_text()); print(ensure_exercise1_analysis(ROOT/'outputs/runs'/s['active_runs']['exercise1']))")
state = json.loads((LAB / "config/workflow.json").read_text())
exercise1_run = LAB / "outputs/runs" / state["active_runs"]["exercise1"]
display(json.loads((exercise1_run / "predictions.json").read_text()))
exercise1_analysis = LAB / "outputs/runs" / state["active_runs"]["exercise1_analysis"]
for image_path in sorted(exercise1_analysis.glob("*.png")):
    display(Markdown(f"**{image_path.name}**"))
    display(Image(filename=str(image_path)))

## P3–P8 — Vòng lặp đầy đủ 10 epoch và regularization

1. Fit TF-IDF trên train; C và threshold baseline chọn trên validation.
2. Mỗi DistilBERT candidate chạy đủ 10 epoch. LR: `[5e-6, 1e-5]`; weight decay 0.1; dropout encoder/attention 0.2, classification head 0.3.
3. Chỉ evaluate và lưu checkpoint **cuối mỗi epoch**. Console tổng kết train-eval loss/accuracy, validation loss/accuracy/Macro-F1 và LR. `epoch_history.json` lưu đúng 10 dòng.
4. Biểu đồ loss/accuracy chính có 10 điểm nguyên từ epoch 1 đến 10. Không lồng validation giữa epoch vào hình này. Train loss chính được đo ở eval mode để so cùng context với validation, có ghi nhãn rõ.
5. Tiêu chí giữ nguyên validation Macro-F1; checkpoint tốt nhất thuộc một epoch nguyên, không bắt buộc epoch 10. Không early stopping.
6. Mỗi candidate không sinh test predictions. Chốt winner/threshold bằng validation rồi mới đánh giá full test.

Dropout/weight decay giảm khả năng học thuộc; LR nhỏ hơn giảm mức cập nhật backbone. Đây là một **gói thay đổi có kiểm soát**, không phải ablation riêng từng yếu tố. Chưa chạy GPU thì chưa có bằng chứng đã giảm overfitting. Không làm mượt hay chỉnh hình để tạo cảm giác loss tốt hơn.

Không fit calibration hoặc label smoothing trong lượt này; giữ công thức CE để dễ diễn giải loss. Test lịch sử đã từng được xem, nên kết quả mới không phải một blind benchmark hoàn toàn mới.

In [ ]:
config = json.loads((LAB / "config/regularized.json").read_text())
study_settings = json.loads((LAB / "config/regularized_study.json").read_text())
assert config["exercise2"]["num_train_epochs"] == 10
assert study_settings["epochs_per_candidate"] == 10
print(json.dumps(study_settings, indent=2))

# Chạy cell này đúng một lần cho mỗi thí nghiệm mới.
# Chạy lại sẽ tạo study và candidate mới, không ghi đè hoặc resume run cũ.
run_python("labs/lab03/scripts/run_study.py", "--config", LAB / "config/regularized.json", "--settings", LAB / "config/regularized_study.json")
state = json.loads((LAB / "config/workflow.json").read_text())
STUDY = LAB / "outputs/runs" / state["active_runs"]["study"]
print("Study đã hoàn tất:", STUDY)

## P8 — Hiển thị toàn bộ kết quả ngay trong notebook

Cell này chạy trong kernel cha bằng IPython.display, không render ảnh trong subprocess. Mỗi candidate hiển thị bảng 10 dòng và learning curves đúng 10 điểm. Dashboard step không hiển thị mặc định; chỉ dùng thêm khi debug optimizer. Các PNG đã có file gốc dưới outputs/runs/ để lấy artifact/report.

Đọc cả train/validation loss, accuracy và Macro-F1. Nếu validation loss vẫn tăng ở các epoch sau, báo cáo trung thực và dùng checkpoint validation tốt nhất, không lấy epoch cuối chỉ vì đã chạy 10 epoch.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("lab03_display_study", LAB / "scripts/display_study.py")
viewer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(viewer)
shown_figures = viewer.display_study(STUDY, root=LAB)
assert len(shown_figures) >= 10, "Thiếu hình trực quan; kiểm tra artifacts trước khi đóng gói."

## Đọc kết quả và thảo luận

1. LR nào thắng trên validation? Checkpoint tốt nhất nằm ở epoch/step nào, có khác epoch 10 không?
2. Từ lúc nào train-eval loss tiếp tục giảm còn validation loss tăng? Có NaN/gradient bất thường không?
3. DistilBERT có hơn TF-IDF trên cùng test không? Mức tăng accuracy và Macro-F1 có cùng chiều không?
4. Threshold làm FP/FN đổi ra sao? Recall giảm có chấp nhận được với mục tiêu bài toán không?
5. ROC-AUC tốt có đồng nghĩa confidence calibrated tốt không? Xem Brier score, NLL và reliability.
6. CI bootstrap phản ánh bất định lấy mẫu test cho một mô hình cố định; không phải CI cho mọi lần train.
7. Có overlap exact/normalized giữa split không? Nếu có, báo cáo rõ, không tự xóa làm thay đổi benchmark.
8. Các câu sai là phủ định, tương phản, mỉa mai hay từ vựng nội dung phim? Giải thích ngôn ngữ là giả thuyết, không khẳng định nguyên nhân mô hình.

Không sửa learning rate, epoch hoặc threshold để tối đa test sau khi xem kết quả. Nếu tiếp tục nghiên cứu, dùng validation và báo cáo việc test đã từng được xem.

## Đóng gói để review và giữ checkpoint

- **Review ZIP**: số liệu, PNG, dự đoán, policy, cấu hình, log nguyên byte, manifest của study và các candidate. Gửi gói này để rà soát và hoàn thiện báo cáo. Không chứa trọng số.
- **Checkpoint ZIP**: chỉ best model của winner, tokenizer, cấu hình và policy. Tải về giữ để inference, reload và phân tích bổ sung mà không phải train lại. Không commit vào Git thông thường.
- Notebook đang mở do Kaggle quản lý: **Save Version**, rồi **File → Download Notebook** để giữ cả output tương tác. Review ZIP không tự lấy được nội dung notebook đang mở từ trình duyệt.

Mọi output gốc nằm dưới `labs/lab03/outputs/`; hai ZIP được sao chép lên `/kaggle/working/` để hiện rõ trong panel Output và tải xuống.

In [ ]:
run_python("labs/lab03/scripts/audit_study.py", "--study-run", STUDY)
run_python("labs/lab03/scripts/package_study.py", "--study-run", STUDY)
bundle = json.loads((LAB / "outputs/bundles/latest_bundle.json").read_text())
for key in ["review", "checkpoint"]:
    if not bundle.get(key):
        continue
    source = Path(bundle[key])
    download = Path("/kaggle/working") / source.name
    shutil.copy2(source, download)
    print(key, ":", download, f"({download.stat().st_size / 1024**2:.1f} MB)")
    display(FileLink(str(download.relative_to(Path.cwd())) if download.is_relative_to(Path.cwd()) else str(download)))
print("SHA256 review:", bundle["review_sha256"])
print("Nếu panel Output chưa hiện ZIP: nhấn refresh cạnh /kaggle/working.")